# 05 - Test uploaded video

Use this notebook first with a local video file you upload into your machine.

It runs the same pipeline used for CCTV:

- YOLO person detection
- ByteTrack tracking
- InsightFace face detection and recognition
- virtual line entry/exit
- SQLite event storage
- annotated output video export


In [ ]:
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'main.py').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root from the current notebook location.')

ROOT = find_project_root()
ROOT


In [ ]:
import sys

sys.path.insert(0, str(ROOT))

from src.utils.config import load_settings
from src.pipeline.runner import run_pipeline

cfg = load_settings(ROOT / 'config' / 'settings.yaml')
print('Using Python:', sys.executable)
print('YOLO weights :', cfg['models']['yolo_weights'])
print('Face root    :', cfg['models']['face_root'])
print('Events DB    :', cfg['events']['db_path'])


In [ ]:
video_path = input('Paste the uploaded video path: ').strip().strip('"').strip("'")
video_path = Path(video_path).expanduser().resolve()
if not video_path.exists():
    raise FileNotFoundError(f'Video not found: {video_path}')

output_dir = ROOT / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f'{video_path.stem}_annotated.mp4'

print('Input :', video_path)
print('Output:', output_path)


In [ ]:
result = run_pipeline(
    cfg,
    source_override=video_path,
    output_path=output_path,
    display=False,
)

print('\nPipeline finished')
print('Frames processed:', result.frames_processed)
print('Events written  :', result.events_count)
print('Annotated video :', result.output_path)


## CCTV next

When the uploaded video works, switch to CCTV by doing one of these:

1. run `python main.py --source rtsp://127.0.0.1:8554/cam_01_sub`
2. or set `camera.source` in `config/settings.yaml` to your RTSP/go2rtc stream
3. or use the same notebook with an RTSP URL instead of a video file
